# EXP-002: Multi-model (XGB + LGB + CAT) + Blending - Playground Series S6E5

**근거:** EXP-001 ES 진단 결과 lr=0.05 기준 OOF 천장 ≈ 0.94915. XGB 단독으로 더 짜기 힘듦 → 모델 다양성 도입.

**전략:**
- 5-Fold StratifiedKFold (seed=42) — 세 모델 **동일 split** 사용해야 OOF 블렌딩 유효
- XGB/LGB는 LabelEncoded cats, CAT는 native categorical 처리
- 학습 설정: 모두 `lr=0.05`, `n_est=2000`, `early_stopping_rounds=200`
- 블렌딩: (a) 단순 평균, (b) OOF 기반 가중치 격자 탐색

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
print(f'Train: {train.shape}, Test: {test.shape}')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
y = train[TARGET].astype(int).values

# LabelEncoded 버전 (XGB / LGB)
train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

# CatBoost 버전 (string 그대로)
train_cb = train.copy(); test_cb = test.copy()
for c in CAT_COLS:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c] = test_cb[c].astype(str)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]
X_cb, X_test_cb = train_cb[feature_cols], test_cb[feature_cols]
print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'Cat indices in feature list: {[feature_cols.index(c) for c in CAT_COLS]}')

# 공통 split — 모든 모델이 같은 OOF index 사용
N_FOLDS, SEED = 5, 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X_le, y))

Train: (439140, 16), Test: (188165, 15)
Features (14): ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']
Cat indices in feature list: [0, 1, 2]


## 1. XGBoost (재학습 — EXP-001과 동일 split)

In [2]:
from xgboost import XGBClassifier

xgb_params = dict(
    n_estimators=2000, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='auc',
    tree_method='hist', random_state=SEED, n_jobs=-1, verbosity=0,
    early_stopping_rounds=200,
)

oof_xgb = np.zeros(len(X_le))
test_xgb = np.zeros(len(X_test_le))
xgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = XGBClassifier(**xgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])], verbose=0)
    oof_xgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_xgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    xgb_iters.append(m.best_iteration)
    print(f'XGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_xgb[va_idx]):.5f}, best_iter={m.best_iteration}')

xgb_oof_auc = roc_auc_score(y, oof_xgb)
print(f'XGB OOF AUC: {xgb_oof_auc:.5f}  (best_iter mean {np.mean(xgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

XGB fold 0: AUC=0.95034, best_iter=932
XGB fold 1: AUC=0.94829, best_iter=819
XGB fold 2: AUC=0.94913, best_iter=854
XGB fold 3: AUC=0.94848, best_iter=858
XGB fold 4: AUC=0.94944, best_iter=861
XGB OOF AUC: 0.94913  (best_iter mean 865, elapsed 61.9s)


## 2. LightGBM

In [3]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgb_params = dict(
    n_estimators=2000, learning_rate=0.05, num_leaves=64, max_depth=-1,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    objective='binary', metric='auc',
    random_state=SEED, n_jobs=-1, verbose=-1,
)

oof_lgb = np.zeros(len(X_le))
test_lgb = np.zeros(len(X_test_le))
lgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = LGBMClassifier(**lgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])],
          callbacks=[early_stopping(200, verbose=False), log_evaluation(0)])
    oof_lgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_lgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    lgb_iters.append(m.best_iteration_)
    print(f'LGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_lgb[va_idx]):.5f}, best_iter={m.best_iteration_}')

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f'LGB OOF AUC: {lgb_oof_auc:.5f}  (best_iter mean {np.mean(lgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

LGB fold 0: AUC=0.95031, best_iter=1822
LGB fold 1: AUC=0.94817, best_iter=1376
LGB fold 2: AUC=0.94908, best_iter=1415
LGB fold 3: AUC=0.94845, best_iter=1548
LGB fold 4: AUC=0.94930, best_iter=1705
LGB OOF AUC: 0.94906  (best_iter mean 1573, elapsed 52.7s)


## 3. CatBoost (native categorical)

In [4]:
from catboost import CatBoostClassifier

cat_idx = [feature_cols.index(c) for c in CAT_COLS]

cb_params = dict(
    iterations=2000, learning_rate=0.05, depth=8, l2_leaf_reg=3.0,
    bagging_temperature=0.5, random_strength=1,
    eval_metric='AUC', loss_function='Logloss',
    cat_features=cat_idx, random_state=SEED,
    verbose=False, allow_writing_files=False, thread_count=-1,
    early_stopping_rounds=200,
)

oof_cat = np.zeros(len(X_cb))
test_cat = np.zeros(len(X_test_cb))
cat_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = CatBoostClassifier(**cb_params)
    m.fit(X_cb.iloc[tr_idx], y[tr_idx],
          eval_set=(X_cb.iloc[va_idx], y[va_idx]),
          use_best_model=True)
    oof_cat[va_idx] = m.predict_proba(X_cb.iloc[va_idx])[:, 1]
    test_cat += m.predict_proba(X_test_cb)[:, 1] / N_FOLDS
    cat_iters.append(m.get_best_iteration())
    print(f'CAT fold {fold}: AUC={roc_auc_score(y[va_idx], oof_cat[va_idx]):.5f}, best_iter={m.get_best_iteration()}')

cat_oof_auc = roc_auc_score(y, oof_cat)
print(f'CAT OOF AUC: {cat_oof_auc:.5f}  (best_iter mean {np.mean(cat_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

CAT fold 0: AUC=0.94992, best_iter=1990
CAT fold 1: AUC=0.94786, best_iter=1999
CAT fold 2: AUC=0.94897, best_iter=1980
CAT fold 3: AUC=0.94825, best_iter=1998
CAT fold 4: AUC=0.94904, best_iter=1996
CAT OOF AUC: 0.94880  (best_iter mean 1993, elapsed 1504.0s)


## 4. Blending

- **단순 평균** (1/3, 1/3, 1/3) — robust, OOF 과적합 위험 낮음
- **OOF 가중치 격자 탐색** — 0.05 step. 보고만 하고 제출은 단순 평균 우선

In [5]:
# (a) 단순 평균
oof_blend_eq  = (oof_xgb + oof_lgb + oof_cat) / 3
test_blend_eq = (test_xgb + test_lgb + test_cat) / 3
auc_eq = roc_auc_score(y, oof_blend_eq)

# (b) OOF 격자 탐색
best = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        w_cat = 1 - w_xgb - w_lgb
        if w_cat < 0: continue
        s = w_xgb * oof_xgb + w_lgb * oof_lgb + w_cat * oof_cat
        a = roc_auc_score(y, s)
        if a > best[1]:
            best = ((round(w_xgb, 2), round(w_lgb, 2), round(w_cat, 2)), a)

w_opt, auc_opt = best
oof_blend_opt  = w_opt[0] * oof_xgb + w_opt[1] * oof_lgb + w_opt[2] * oof_cat
test_blend_opt = w_opt[0] * test_xgb + w_opt[1] * test_lgb + w_opt[2] * test_cat

print('=== 결과 ===')
print(f'XGB         OOF AUC: {xgb_oof_auc:.5f}')
print(f'LGB         OOF AUC: {lgb_oof_auc:.5f}')
print(f'CAT         OOF AUC: {cat_oof_auc:.5f}')
print(f'Blend equal       :  {auc_eq:.5f}')
print(f'Blend opt {w_opt}: {auc_opt:.5f}  ← 위험: OOF 과적합')

=== 결과 ===
XGB         OOF AUC: 0.94913
LGB         OOF AUC: 0.94906
CAT         OOF AUC: 0.94880
Blend equal       :  0.95010
Blend opt (np.float64(0.3), np.float64(0.3), np.float64(0.4)): 0.95011  ← 위험: OOF 과적합


## 5. Submissions

In [6]:
import os
os.makedirs('../submissions', exist_ok=True)

for name, prob in [
    ('exp002_xgb',         test_xgb),
    ('exp002_lgb',         test_lgb),
    ('exp002_cat',         test_cat),
    ('exp002_blend_equal', test_blend_eq),
    ('exp002_blend_opt',   test_blend_opt),
]:
    sub = pd.DataFrame({'id': test['id'], 'PitNextLap': prob})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'  saved {path}  mean={prob.mean():.4f}')

  saved ../submissions/submission_exp002_xgb.csv  mean=0.1974
  saved ../submissions/submission_exp002_lgb.csv  mean=0.1972
  saved ../submissions/submission_exp002_cat.csv  mean=0.1983
  saved ../submissions/submission_exp002_blend_equal.csv  mean=0.1976
  saved ../submissions/submission_exp002_blend_opt.csv  mean=0.1977
